# ============================================================
# MODELO HÍBRIDO EIF — PREPROCESADO FFT
# Modelo eléctrico : u, v, w × 1001 bins = 3003 features
# Modelo vibración : 5 señales × 1001 bins = 5005 features
# Decisión final   : OR lógico (detectado si cualquiera de los dos supera umbral)
# Umbral           : media + 3·std por modelo (paper)
# ============================================================

In [ ]:
# ============================================================
# 0. INSTALACIÓN
# ============================================================

!pip install h2o optuna optuna-dashboard plotly seaborn

In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import h2o
from h2o.estimators import H2OExtendedIsolationForestEstimator
import optuna
from optuna.trial import TrialState
import optuna.visualization as vis
import numpy as np
import pandas as pd
import os
import json
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.io as pio
pio.renderers.default = "browser"

In [ ]:
# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

RUTA_FEATURES   = "features_fft"
CSV_INDEX       = "Motor_DB/index/master_index.csv"
RUTA_RESULTADOS = "resultados_fft_hibrido"
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

N_TRIALS         = 500
SEED             = 42
VENTANAS_POR_EXP = 100   # 100 segundos a 20kHz, ventanas de 1s
VELOCIDADES_ESTABLES = {"S1500", "S1200", "S900"}  # regímenes estables (excluye STrap, SSteps, SStart y S20)


In [ ]:
# ============================================================
# 3. LEER COLUMNAS Y CALCULAR EXT_MAX POR MODELO
# ============================================================

cols_todas      = pd.read_csv(os.path.join(RUTA_FEATURES, "train", "sano_train.csv"), nrows=0).columns.tolist()
cols_electricas = [c for c in cols_todas if c.startswith(("u_bin", "v_bin", "w_bin"))]
cols_vibracion  = [c for c in cols_todas if c not in cols_electricas]

# EXT_MAX real para cada submodelo
EXT_MAX_ELEC_REAL = len(cols_electricas) - 1   # 3002
EXT_MAX_VIB_REAL  = len(cols_vibracion)  - 1   # 5004

# Limitar búsqueda Optuna (valores altos ralentizan mucho)
EXT_MAX_ELEC = min(EXT_MAX_ELEC_REAL, 100)
EXT_MAX_VIB  = min(EXT_MAX_VIB_REAL,  100)

print(f"Eléctricas : {len(cols_electricas)} features  (EXT_MAX búsqueda: {EXT_MAX_ELEC})")
print(f"Vibración  : {len(cols_vibracion)} features  (EXT_MAX búsqueda: {EXT_MAX_VIB})")

In [ ]:
# ============================================================
# 4. INICIALIZAR H2O
# ============================================================

h2o.init(nthreads=-1, max_mem_size="16G")

In [ ]:
# ============================================================
# 5. CARGAR DATOS Y SEPARAR POR MODELO
# ============================================================

print("Cargando datos...\n")

index        = pd.read_csv(CSV_INDEX)

# Filtrar transitorios — mismo criterio que el preprocesado FFT
index = index[index["Velocidad"].astype(str).isin(VELOCIDADES_ESTABLES)].reset_index(drop=True)
print(f"Index tras filtro: {len(index)} experimentos (transitorios excluidos)\n")
carpeta_test = os.path.join(RUTA_FEATURES, "test")

# Frames completos
train_full     = h2o.import_file(os.path.join(RUTA_FEATURES, "train", "sano_train.csv"))
val_full       = h2o.import_file(os.path.join(RUTA_FEATURES, "val",   "sano.csv"))
test_sano_full = h2o.import_file(os.path.join(RUTA_FEATURES, "test",  "sano.csv"))

fallos_full = {}
for archivo in sorted(os.listdir(carpeta_test)):
    if archivo.endswith(".csv") and archivo != "sano.csv":
        nombre = archivo.replace(".csv", "")
        fallos_full[nombre] = h2o.import_file(os.path.join(carpeta_test, archivo))

# Separar columnas por submodelo
train_elec     = train_full[cols_electricas]
train_vib      = train_full[cols_vibracion]
val_elec       = val_full[cols_electricas]
val_vib        = val_full[cols_vibracion]
test_sano_elec = test_sano_full[cols_electricas]
test_sano_vib  = test_sano_full[cols_vibracion]

fallos_elec = {n: f[cols_electricas] for n, f in fallos_full.items()}
fallos_vib  = {n: f[cols_vibracion]  for n, f in fallos_full.items()}

print(f"Train eléctrico : {train_elec.shape}")
print(f"Train vibración : {train_vib.shape}")
print(f"Grupos de fallo : {len(fallos_full)}")

In [ ]:
# ============================================================
# 6. FUNCIONES AUXILIARES
# ============================================================

def agregar_por_experimento(scores_array, ventanas_por_exp):
    medianas = []
    for i in range(0, len(scores_array), ventanas_por_exp):
        grupo = scores_array[i : i + ventanas_por_exp]
        if len(grupo) > 0:
            medianas.append(np.median(grupo))
    return np.array(medianas)

def agregar_variable(scores_array, n_archivos):
    ventanas_por_exp = len(scores_array) // n_archivos
    return agregar_por_experimento(scores_array, ventanas_por_exp)

def calcular_umbral(model, train_frame, val_frame):
    s_t   = model.predict(train_frame)["anomaly_score"].as_data_frame().values.flatten()
    s_v   = model.predict(val_frame)["anomaly_score"].as_data_frame().values.flatten()
    med_t = agregar_por_experimento(s_t, VENTANAS_POR_EXP)
    med_v = agregar_por_experimento(s_v, VENTANAS_POR_EXP)
    todos = np.concatenate([med_t, med_v])
    media = np.mean(todos)
    std   = np.std(todos)
    return media + 3 * std, media, std, med_t, med_v

In [ ]:
# ============================================================
# 7. OPTUNA — MODELO ELÉCTRICO
# ============================================================

print("\n" + "="*60)
print("OPTIMIZANDO MODELO ELÉCTRICO")
print(f"  Features: {len(cols_electricas)}  |  EXT_MAX: {EXT_MAX_ELEC}")
print("="*60)

def objective_elec(trial):
    ntrees          = trial.suggest_int("ntrees",          100, 800)
    sample_size     = trial.suggest_int("sample_size",     256, 1024, step=64)
    extension_level = trial.suggest_int("extension_level", 0,   EXT_MAX_ELEC)

    model = H2OExtendedIsolationForestEstimator(
        ntrees=ntrees, sample_size=sample_size,
        extension_level=extension_level, seed=SEED
    )
    try:
        model.train(training_frame=train_elec)
    except Exception:
        raise optuna.exceptions.TrialPruned()

    s_t   = model.predict(train_elec)["anomaly_score"].as_data_frame().values.flatten()
    s_v   = model.predict(val_elec)["anomaly_score"].as_data_frame().values.flatten()
    med_t = agregar_por_experimento(s_t, VENTANAS_POR_EXP)
    med_v = agregar_por_experimento(s_v, VENTANAS_POR_EXP)
    todos = np.concatenate([med_t, med_v])
    umbral = np.mean(todos) + 3 * np.std(todos)
    trial.set_user_attr("umbral", round(float(umbral), 6))
    return umbral

study_elec = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
    storage=f"sqlite:///{RUTA_RESULTADOS}/optuna_elec.db",
    study_name="EIF_electrico_fft", load_if_exists=True
)
study_elec.optimize(objective_elec, n_trials=N_TRIALS, show_progress_bar=True)

In [ ]:
# ============================================================
# 8. OPTUNA — MODELO VIBRACIÓN
# ============================================================

print("\n" + "="*60)
print("OPTIMIZANDO MODELO VIBRACIÓN")
print(f"  Features: {len(cols_vibracion)}  |  EXT_MAX: {EXT_MAX_VIB}")
print("="*60)

def objective_vib(trial):
    ntrees          = trial.suggest_int("ntrees",          100, 800)
    sample_size     = trial.suggest_int("sample_size",     256, 1024, step=64)
    extension_level = trial.suggest_int("extension_level", 0,   EXT_MAX_VIB)

    model = H2OExtendedIsolationForestEstimator(
        ntrees=ntrees, sample_size=sample_size,
        extension_level=extension_level, seed=SEED
    )
    try:
        model.train(training_frame=train_vib)
    except Exception:
        raise optuna.exceptions.TrialPruned()

    s_t   = model.predict(train_vib)["anomaly_score"].as_data_frame().values.flatten()
    s_v   = model.predict(val_vib)["anomaly_score"].as_data_frame().values.flatten()
    med_t = agregar_por_experimento(s_t, VENTANAS_POR_EXP)
    med_v = agregar_por_experimento(s_v, VENTANAS_POR_EXP)
    todos = np.concatenate([med_t, med_v])
    umbral = np.mean(todos) + 3 * np.std(todos)
    trial.set_user_attr("umbral", round(float(umbral), 6))
    return umbral

study_vib = optuna.create_study(
    direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED),
    storage=f"sqlite:///{RUTA_RESULTADOS}/optuna_vib.db",
    study_name="EIF_vibracion_fft", load_if_exists=True
)
study_vib.optimize(objective_vib, n_trials=N_TRIALS, show_progress_bar=True)

In [ ]:
# ============================================================
# 9. MODELOS FINALES
# ============================================================

# Reiniciar H2O para liberar memoria
h2o.cluster().shutdown(prompt=False)
import time; time.sleep(5)
h2o.init(nthreads=-1, max_mem_size="16G")

# Recargar datos
train_full     = h2o.import_file(os.path.join(RUTA_FEATURES, "train", "sano_train.csv"))
val_full       = h2o.import_file(os.path.join(RUTA_FEATURES, "val",   "sano.csv"))
test_sano_full = h2o.import_file(os.path.join(RUTA_FEATURES, "test",  "sano.csv"))
fallos_full    = {}
for archivo in sorted(os.listdir(carpeta_test)):
    if archivo.endswith(".csv") and archivo != "sano.csv":
        fallos_full[archivo.replace(".csv","")] = h2o.import_file(os.path.join(carpeta_test, archivo))

train_elec     = train_full[cols_electricas];  train_vib      = train_full[cols_vibracion]
val_elec       = val_full[cols_electricas];    val_vib        = val_full[cols_vibracion]
test_sano_elec = test_sano_full[cols_electricas]; test_sano_vib = test_sano_full[cols_vibracion]
fallos_elec    = {n: f[cols_electricas] for n, f in fallos_full.items()}
fallos_vib     = {n: f[cols_vibracion]  for n, f in fallos_full.items()}

# Recargar estudios Optuna
study_elec = optuna.load_study(study_name="EIF_electrico_fft", storage=f"sqlite:///{RUTA_RESULTADOS}/optuna_elec.db")
study_vib  = optuna.load_study(study_name="EIF_vibracion_fft", storage=f"sqlite:///{RUTA_RESULTADOS}/optuna_vib.db")
best_elec  = study_elec.best_trial
best_vib   = study_vib.best_trial

print("\n" + "="*60)
print("ENTRENANDO MODELOS FINALES")
print("="*60)
print(f"\nEléctrico → ntrees={best_elec.params['ntrees']}, "
      f"sample_size={best_elec.params['sample_size']}, "
      f"extension_level={best_elec.params['extension_level']}")
print(f"Vibración → ntrees={best_vib.params['ntrees']}, "
      f"sample_size={best_vib.params['sample_size']}, "
      f"extension_level={best_vib.params['extension_level']}")

modelo_elec = H2OExtendedIsolationForestEstimator(
    ntrees=best_elec.params["ntrees"], sample_size=best_elec.params["sample_size"],
    extension_level=best_elec.params["extension_level"], seed=SEED)
modelo_elec.train(training_frame=train_elec)

modelo_vib = H2OExtendedIsolationForestEstimator(
    ntrees=best_vib.params["ntrees"], sample_size=best_vib.params["sample_size"],
    extension_level=best_vib.params["extension_level"], seed=SEED)
modelo_vib.train(training_frame=train_vib)

umbral_elec, media_e, std_e, med_train_e, med_val_e = calcular_umbral(modelo_elec, train_elec, val_elec)
umbral_vib,  media_v, std_v, med_train_v, med_val_v = calcular_umbral(modelo_vib,  train_vib,  val_vib)

print(f"\nUmbral eléctrico : {umbral_elec:.6f}  (media={media_e:.4f}, std={std_e:.4f})")
print(f"Umbral vibración : {umbral_vib:.6f}   (media={media_v:.4f}, std={std_v:.4f})")

with open(os.path.join(RUTA_RESULTADOS, "hiperparametros.json"), "w") as f:
    json.dump({
        "electrico": best_elec.params, "vibracion": best_vib.params,
        "umbral_elec": umbral_elec, "umbral_vib": umbral_vib
    }, f, indent=2)

In [ ]:
# ============================================================
# 10. EVALUACIÓN POR EXPERIMENTO
# ============================================================

print("\n" + "="*60)
print("EVALUACIÓN POR EXPERIMENTO")
print("="*60)

resultados  = []
datos_box_e = []
datos_box_v = []

def evaluar(nombre, frame_e, frame_v, n_archivos, es_fallo=True):
    s_e   = modelo_elec.predict(frame_e)["anomaly_score"].as_data_frame().values.flatten()
    s_v   = modelo_vib.predict(frame_v)["anomaly_score"].as_data_frame().values.flatten()
    med_e = agregar_variable(s_e, n_archivos)
    med_v = agregar_variable(s_v, n_archivos)

    det_e    = med_e > umbral_elec
    det_v    = med_v > umbral_vib
    det_comb = det_e | det_v   # OR lógico: fallo si cualquiera supera su umbral

    for m in med_e: datos_box_e.append({"grupo": nombre, "mediana": m, "es_fallo": es_fallo})
    for m in med_v: datos_box_v.append({"grupo": nombre, "mediana": m, "es_fallo": es_fallo})

    return {
        "grupo":            nombre,
        "n_exp":            len(med_e),
        "det_elec_%":       round(np.mean(det_e)    * 100, 1),
        "det_vib_%":        round(np.mean(det_v)    * 100, 1),
        "det_combinado_%":  round(np.mean(det_comb) * 100, 1),
    }

# Añadir train/val al boxplot
for nombre, med_e, med_v in [
    ("sano_train", med_train_e, med_train_v),
    ("sano_val",   med_val_e,   med_val_v)
]:
    for m in med_e: datos_box_e.append({"grupo": nombre, "mediana": m, "es_fallo": False})
    for m in med_v: datos_box_v.append({"grupo": nombre, "mediana": m, "es_fallo": False})

# Sano test
n_sano = len(index[(index["Split"] == "test") & (index["Maquina"] == "h")])
res    = evaluar("sano_test", test_sano_elec, test_sano_vib, n_sano, es_fallo=False)
resultados.append(res)
print(f"\n  sano_test → elec: {res['det_elec_%']}% | vib: {res['det_vib_%']}% | "
      f"combinado: {res['det_combinado_%']}%  ← idealmente 0%")

# Fallos
print()
for nombre in sorted(fallos_elec.keys()):
    n_arch = len(index[(index["Split"] == "test") & (index["Fallo"] == nombre)])
    res    = evaluar(nombre, fallos_elec[nombre], fallos_vib[nombre], n_arch)
    resultados.append(res)
    print(f"  {nombre:<55}  "
          f"elec: {res['det_elec_%']:>5.1f}%  "
          f"vib: {res['det_vib_%']:>5.1f}%  "
          f"combinado: {res['det_combinado_%']:>5.1f}%")

df_res = pd.DataFrame(resultados)
df_res.to_csv(os.path.join(RUTA_RESULTADOS, "resultados_hibrido.csv"), index=False)
print(f"\nGuardado: {RUTA_RESULTADOS}/resultados_hibrido.csv")

In [ ]:
# ============================================================
# 11. BOXPLOTS
# ============================================================

grupos_sanos  = ["sano_train", "sano_val", "sano_test"]
grupos_fallos = sorted([n for n in fallos_elec.keys()])
orden         = grupos_sanos + grupos_fallos
colores       = {g: "steelblue" if g in grupos_sanos else "salmon" for g in orden}

def hacer_boxplot(datos, umbral_linea, titulo, filename):
    df_b = pd.DataFrame(datos)
    plt.figure(figsize=(24, 7))
    sns.boxplot(
        data    = df_b[df_b["grupo"].isin(orden)],
        x       = "grupo",
        y       = "mediana",
        order   = orden,
        palette = colores
    )
    plt.axhline(y=umbral_linea, color="red", linestyle="--", linewidth=1.5,
                label=f"Umbral = {umbral_linea:.4f}")
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.title(titulo)
    plt.xlabel("Grupo")
    plt.ylabel("Mediana Anomaly Score")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(RUTA_RESULTADOS, filename), dpi=150)
    plt.show()

hacer_boxplot(datos_box_e, umbral_elec,
              "Modelo Eléctrico FFT (u, v, w — 3003 features)",
              "boxplot_electrico.png")

hacer_boxplot(datos_box_v, umbral_vib,
              "Modelo Vibración FFT (front_DE, rear_NDE, housing — 5005 features)",
              "boxplot_vibracion.png")

print("\n✅ Todo completado.")
print(f"Umbral eléctrico : {umbral_elec:.6f}")
print(f"Umbral vibración : {umbral_vib:.6f}")

In [ ]:
# ============================================================
# 12. GRÁFICAS OPTUNA
# ============================================================

for study, nombre in [(study_elec, "electrico"), (study_vib, "vibracion")]:
    print(f"\nGráficas Optuna — {nombre}")

    fig1 = vis.plot_optimization_history(study)
    fig1.update_layout(title=f"Historia optimización — {nombre}")
    fig1.write_html(os.path.join(RUTA_RESULTADOS, f"optuna_{nombre}_historia.html"))
    fig1.show()

    fig2 = vis.plot_param_importances(study)
    fig2.update_layout(title=f"Importancia hiperparámetros — {nombre}")
    fig2.write_html(os.path.join(RUTA_RESULTADOS, f"optuna_{nombre}_importancia.html"))
    fig2.show()

    fig3 = vis.plot_contour(study, params=["ntrees", "extension_level"])
    fig3.update_layout(title=f"Contour ntrees vs extension_level — {nombre}")
    fig3.write_html(os.path.join(RUTA_RESULTADOS, f"optuna_{nombre}_contour.html"))
    fig3.show()

print("\n✅ Todo completado.")
h2o.cluster().show_status()